In [14]:
%load_ext autoreload
%autoreload 2
# %flow mode reactive

import datetime
import os
import warnings
from pathlib import Path
from typing import Any, Tuple, List, Dict

import numpy as np
import pandas as pd
import plotly
import plotly.express as px
import plotly.graph_objs as go
import statsmodels.api as sm
from plotly.subplots import make_subplots
from tqdm.notebook import tqdm

import datajoint as dj
from aeon.dj_pipeline.analysis.block_analysis import *
from aeon.dj_pipeline import acquisition, streams
from swc.aeon.io import api as aeon_api

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Missing data:
- Directly queryable from db
  - Centroid and ID tracking over the full period of time
  - Foraging bouts
- To be generated from queried data
  - Sleeping bouts
  - Exploring bouts

# Load data

In [5]:
experiments = [
    {"name": "social0.2-aeon3", "pre_social_start": '2024-01-31 11:00:00', "pre_social_end": '2024-02-08 15:00:00', "social_start": '2024-02-09 16:00:00', "social_end": '2024-02-23 13:00:00', "post_social_start": '2024-02-25 16:00:00', "post_social_end": '2024-03-02 14:00:00'},
    {"name": "social0.2-aeon4", "pre_social_start": '2024-01-31 10:00:00', "pre_social_end": '2024-02-08 15:00:00', "social_start": '2024-02-09 16:00:00', "social_end": '2024-02-23 12:00:00', "post_social_start": '2024-02-25 16:00:00', "post_social_end": '2024-03-02 13:00:00'},
    {"name": "social0.3-aeon3", "pre_social_start": '2024-06-08 18:00:00', "pre_social_end": '2024-06-17 13:00:00', "social_start": '2024-06-25 10:00:00', "social_end": '2024-07-06 13:00:00', "post_social_start": '2024-07-07 15:00:00', "post_social_end": '2024-07-14 14:00:00'},
    {"name": "social0.3-aeon4", "pre_social_start": '2024-06-08 18:00:00', "pre_social_end": '2024-06-17 14:00:00', "social_start": '2024-06-19 11:00:00', "social_end": '2024-07-03 14:00:00', "post_social_start": '2024-07-04 10:00:00', "post_social_end": '2024-07-13 12:00:00'},
    {"name": "social0.4-aeon3", "pre_social_start": '2024-08-16 16:00:00', "pre_social_end": '2024-08-24 10:00:00', "social_start": '2024-08-28 10:00:00', "social_end": '2024-09-09 13:00:00', "post_social_start": '2024-09-09 17:00:00', "post_social_end": '2024-09-22 16:00:00'},
    {"name": "social0.4-aeon4", "pre_social_start": '2024-08-16 14:00:00', "pre_social_end": '2024-08-24 10:00:00', "social_start": '2024-08-28 09:00:00', "social_end": '2024-09-09 01:00:00', "post_social_start": '2024-09-09 14:00:00', "post_social_end": '2024-09-22 16:00:00'}
]

## Patch data

In [ ]:
def load_subject_patch_data(
    key: dict[str, str],
    period_start: str,
    period_end: str
) -> tuple[list[dict[str, str]], pd.DataFrame]:
    """Loads subject patch data for a specified time period.

    Args:
        key (dict): The key to filter the subject patch data.
        period_start (str): The start time for the period.
        period_end (str): The end time for the period.

    Returns:
        tuple: A tuple containing:
            - patch_info (list of dict): Information about patches.
            - block_subject_patch_data (pd.DataFrame): Data for the specified period.
    """
    patch_info = (
        BlockAnalysis.Patch
        & key
        & f'block_start >= "{period_start}"'
        & f'block_start <= "{period_end}"'
    ).fetch('block_start', "patch_name", "patch_rate", "patch_offset", as_dict=True)

    block_subject_patch_data = (
        BlockSubjectAnalysis.Patch() 
        & key 
        & f'block_start >= "{period_start}"' 
        & f'block_start <= "{period_end}"'
    ).fetch(format="frame")

    if not block_subject_patch_data.empty:
        block_subject_patch_data.reset_index(level=["experiment_name"], drop=True, inplace=True) 
        block_subject_patch_data.reset_index(inplace=True)

    return patch_info, block_subject_patch_data

In [4]:
patch_info_dict = {}
subject_patch_data_dict = {}

for exp in experiments:
    key = {"experiment_name": exp["name"]}

    # Define periods
    periods = {
        "pre_social": (exp["pre_social_start"], exp["pre_social_end"]),
        "social": (exp["social_start"], exp["social_end"]),
        "post_social": (exp["post_social_start"], exp["post_social_end"])
    }

    # Initialize nested dictionaries for this experiment
    patch_info_dict[exp["name"]] = {}
    subject_patch_data_dict[exp["name"]] = {}

    # Load data for each period
    for period_name, (period_start, period_end) in periods.items():
        # Convert string dates to datetime if needed
        period_start = datetime.strptime(period_start, "%Y-%m-%d %H:%M:%S")
        period_end = datetime.strptime(period_end, "%Y-%m-%d %H:%M:%S")

        # Load data for this period
        patch_info, block_subject_patch_data = load_subject_patch_data(
            key, period_start, period_end
        )

        # Filter out dummy patches
        if not block_subject_patch_data.empty:
            block_subject_patch_data = block_subject_patch_data[
                ~block_subject_patch_data["patch_name"].str.contains("PatchDummy")
            ]

            # Add experiment name as a column
            block_subject_patch_data.insert(0, "experiment_name", exp["name"])

            # For pre-social and post-social periods check n_subjects per block (should == 1)
            if period_name in ["pre_social", "post_social"]:
                n_subjects = (
                    block_subject_patch_data.groupby("block_start")["subject_name"].nunique()
                )
                if (n_subjects != 1).any():
                    warnings.warn(
                        f"Pre or post social data for {exp['name']} has blocks with more than one "
                        f"subject being tracked. Data needs to be fixed or cleaned."
                    )

        # Store the data
        patch_info_dict[exp["name"]][period_name] = patch_info
        subject_patch_data_dict[exp["name"]][period_name] = block_subject_patch_data

# Combine data across experiments for each period
combined_data = {}
for period_name in ["pre_social", "social", "post_social"]:
    period_data = []
    for exp_name in patch_info_dict:
        if not subject_patch_data_dict[exp_name][period_name].empty:
            period_data.append(subject_patch_data_dict[exp_name][period_name])

    if period_data:
        combined_data[period_name] = pd.concat(period_data)
    else:
        # Create empty DataFrame with expected columns if no data
        combined_data[period_name] = pd.DataFrame()

# Assign to the original variable names for compatibility
block_subject_patch_data_pre_social_combined = combined_data["pre_social"]
block_subject_patch_data_social_combined = combined_data["social"]
block_subject_patch_data_post_social_combined = combined_data["post_social"]

# Display one of the dataframes as an example
block_subject_patch_data_social_combined

,experiment_name,block_start,patch_name,subject_name,in_patch_timestamps,in_patch_time,in_patch_rfid_timestamps,pellet_count,pellet_timestamps,patch_threshold,wheel_cumsum_distance_travelled
0,social0.2-aeon3,2024-02-09 17:44:52.000,Patch1,BAA-1104045,"[2024-02-09T17:56:07.700000000, 2024-02-09T17:...",215.1,"[2024-02-09T17:59:38.312480000, 2024-02-09T17:...",2,"[2024-02-09T18:49:53.179488000, 2024-02-09T18:...","[1464.8411046692918, 498.76823784586446]","[-0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -0.004602..."
1,social0.2-aeon3,2024-02-09 17:44:52.000,Patch1,BAA-1104047,"[2024-02-09T17:56:02.800000000, 2024-02-09T17:...",180.3,"[2024-02-09T17:56:07.631712000, 2024-02-09T17:...",2,"[2024-02-09T19:06:36.652480000, 2024-02-09T19:...","[442.1824932452, 803.7255692660416]","[0.0, 0.0, -0.004602223261073846, 0.0030681488..."
2,social0.2-aeon3,2024-02-09 17:44:52.000,Patch2,BAA-1104045,"[2024-02-09T17:58:37.600000000, 2024-02-09T17:...",505.0,"[2024-02-09T18:00:17.962400000, 2024-02-09T18:...",6,"[2024-02-09T18:42:08.057504000, 2024-02-09T19:...","[205.67398184701912, 2831.169369443016, 784.51...","[-0.0, 0.00153407442035558, 0.0015340744203555..."
3,social0.2-aeon3,2024-02-09 17:44:52.000,Patch2,BAA-1104047,"[2024-02-09T17:55:44.100000000, 2024-02-09T17:...",462.9,"[2024-02-09T18:04:16.902368000, 2024-02-09T18:...",7,"[2024-02-09T18:06:38.471488000, 2024-02-09T18:...","[297.1758398118849, 138.93143165562867, 1139.0...","[0.0, 0.0, -0.00153407442035558, -0.0061362976..."
4,social0.2-aeon3,2024-02-09 17:44:52.000,Patch3,BAA-1104045,"[2024-02-09T17:52:31.700000000, 2024-02-09T17:...",676.7,"[2024-02-09T17:52:33.001984000, 2024-02-09T17:...",10,"[2024-02-09T17:53:19.379488000, 2024-02-09T17:...","[520.6659443864802, 183.26738749049315, 955.30...","[-0.0, -0.0015340744203591328, -0.001534074420..."
...,...,...,...,...,...,...,...,...,...,...,...
1133,social0.4-aeon4,2024-09-09 00:00:01.012,Patch1,BAA-1104797,[],0.0,[],0,[],[],"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1134,social0.4-aeon4,2024-09-09 00:00:01.012,Patch2,BAA-1104795,[],0.0,[],0,[],[],"[-0.0, -0.0015340744203582446, -0.003068148840..."
1135,social0.4-aeon4,2024-09-09 00:00:01.012,Patch2,BAA-1104797,[],0.0,[],0,[],[],"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1136,social0.4-aeon4,2024-09-09 00:00:01.012,Patch3,BAA-1104795,[],0.0,[],0,[],[],"[-0.0, 0.0, -0.0015340744203591328, -0.0046022..."


### Foraging bouts

In [10]:
def load_foraging_bouts(
    key: Dict[str, str],
    period_start: str,
    period_end: str
) -> pd.DataFrame:
    """Loads foraging bout data for blocks falling within a specified time period.

    Args:
        key (dict): Key to identify experiment data (e.g., {"experiment_name": "Exp1"}).
        period_start (str): Start datetime of the time period (format: '%Y-%m-%d %H:%M:%S').
        period_end (str): End datetime of the time period (format: '%Y-%m-%d %H:%M:%S').

    Returns:
        pd.DataFrame: Concatenated dataframe of foraging bouts for all matching blocks.
                      Returns an empty dataframe with predefined columns if no data found.
    """
    # Fetch block start times within the specified period
    blocks = (
        Block
        & key
        & f"block_start >= '{period_start}'"
        & f"block_end <= '{period_end}'"
    ).fetch("block_start")

    # Retrieve foraging bouts for each block
    bouts = []
    for block_start in blocks:
        block_key = key | {"block_start": str(block_start)}
        bouts.append(get_foraging_bouts(block_key))

    # Return concatenated DataFrame or empty fallback
    if bouts:
        return pd.concat(bouts, ignore_index=True)
    else:
        return pd.DataFrame(
            columns=["start", "end", "n_pellets", "cum_wheel_dist", "subject"]
        )

In [11]:
foraging_pre_social_dict = {}
foraging_social_dict = {}
foraging_post_social_dict = {}

for exp in experiments:
    key = {"experiment_name": exp["name"]}

    # Load foraging bout data for each time period
    pre_df = load_foraging_bouts(key, exp["pre_social_start"], exp["pre_social_end"])
    social_df = load_foraging_bouts(key, exp["social_start"], exp["social_end"])
    post_df = load_foraging_bouts(key, exp["post_social_start"], exp["post_social_end"])

    # Add experiment name as a column
    pre_df.insert(0, "experiment_name", exp["name"])
    social_df.insert(0, "experiment_name", exp["name"])
    post_df.insert(0, "experiment_name", exp["name"])

    # Store individual DataFrames in dictionaries
    foraging_pre_social_dict[exp["name"]] = pre_df
    foraging_social_dict[exp["name"]] = social_df
    foraging_post_social_dict[exp["name"]] = post_df

# Combine all foraging bout data
foraging_pre_social = pd.concat(foraging_pre_social_dict.values(), ignore_index=True)
foraging_social = pd.concat(foraging_social_dict.values(), ignore_index=True)
foraging_post_social = pd.concat(foraging_post_social_dict.values(), ignore_index=True)

# Final formatting
for df in (foraging_pre_social, foraging_social, foraging_post_social):
    df.sort_values(["experiment_name", "start"], inplace=True)
    df.reset_index(drop=True, inplace=True)

# Display an example
foraging_social

/nfs/nhome/live/jbhagat/ProjectAeon/aeon_mecha/aeon/dj_pipeline/analysis/block_analysis.py:1926: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  bout_data = pd.concat(
/nfs/nhome/live/jbhagat/ProjectAeon/aeon_mecha/aeon/dj_pipeline/analysis/block_analysis.py:1926: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  bout_data = pd.concat(
/nfs/nhome/live/jbhagat/ProjectAeon/aeon_mecha/aeon/dj_pipeline/analysis/block_analysis.py:1926: FutureWarning: The behavior of DataFrame concatenation with empty or al

,experiment_name,start,end,n_pellets,cum_wheel_dist,subject
0,social0.2-aeon3,2024-02-09 18:49:43.300,2024-02-09 18:53:06.800,3,1697.691128,BAA-1104045
1,social0.2-aeon3,2024-02-09 19:04:41.000,2024-02-09 19:08:31.300,3,1546.017190,BAA-1104047
2,social0.2-aeon3,2024-02-09 19:08:35.220,2024-02-09 19:11:01.680,4,1212.780942,BAA-1104047
3,social0.2-aeon3,2024-02-09 19:38:43.900,2024-02-09 19:44:52.660,8,3936.660472,BAA-1104047
4,social0.2-aeon3,2024-02-09 19:48:06.020,2024-02-09 19:55:15.500,5,2519.732576,BAA-1104045
...,...,...,...,...,...,...
2657,social0.4-aeon4,2024-09-08 17:28:34.400,2024-09-08 17:31:10.300,3,871.635006,BAA-1104795
2658,social0.4-aeon4,2024-09-08 17:55:46.400,2024-09-08 18:01:57.400,5,1776.298635,BAA-1104797
2659,social0.4-aeon4,2024-09-08 17:59:12.900,2024-09-08 18:04:08.600,4,1723.506532,BAA-1104795
2660,social0.4-aeon4,2024-09-08 18:09:38.900,2024-09-08 18:13:57.400,5,1316.800392,BAA-1104795


## Position data

In [12]:
def load_position_data(
    key: Dict[str, str],
    period_start: str,
    period_end: str
) -> pd.DataFrame:
    """Loads position data (centroid tracking) for a specified time period.

    Args:
        key (dict): Key to identify experiment data (e.g., {"experiment_name": "Exp1"}).
        period_start (str): Start datetime of the time period.
        period_end (str): End datetime of the time period.

    Returns:
        pd.DataFrame: DataFrame containing position data for the specified period.
                     Returns an empty DataFrame if no data found.
    """
    try:
        print(f"  Querying data from {period_start} to {period_end}...")

        # Create chunk restriction for the time period
        chunk_restriction = acquisition.create_chunk_restriction(
            key["experiment_name"], period_start, period_end
        )

        # Create the query
        pose_query = (
            streams.SpinnakerVideoSource
            * tracking.SLEAPTracking.PoseIdentity.proj(
                "identity_name", "identity_likelihood", "anchor_part"
            )
            * tracking.SLEAPTracking.AnchorPart
            & key
            & {
                "spinnaker_video_source_name": "CameraTop",
            }
            & chunk_restriction
        )

        # Fetch the data
        centroid_df = fetch_stream(pose_query)

        # Clean up the dataframe
        if not centroid_df.empty:
            if "spinnaker_video_source_name" in centroid_df.columns:
                centroid_df.drop(columns=["spinnaker_video_source_name"], inplace=True)

            # Add experiment name column for reference
            centroid_df.insert(0, "experiment_name", key["experiment_name"])

            print(f"  Retrieved {len(centroid_df)} rows of position data")
        else:
            print("  No data found for the specified period")

        return centroid_df

    except Exception as e:
        print(
            f"  Error loading position data for {key['experiment_name']} ({period_start} "
            f"to {period_end}): {e}"
        )
        return pd.DataFrame()  # Empty DataFrame

def save_position_data_to_parquet(
    df: pd.DataFrame,
    experiment_name: str,
    period_name: str,
    data_dir: Path
) -> Path:
    """Saves position data DataFrame to a parquet file.

    Args:
        df (pd.DataFrame): Position data to save
        experiment_name (str): Name of the experiment
        period_name (str): Period name (pre_social, social, post_social)
        data_dir (Path): Directory to save the file

    Returns:
        Path: Path to the saved file
    """
    # Create directory if it doesn't exist
    os.makedirs(data_dir, exist_ok=True)

    # Add period column for reference
    df = df.copy()
    df = df.reset_index()
    df["period"] = period_name

    # Create filename
    filename = f"{experiment_name}_{period_name}_position.parquet"
    file_path = data_dir / filename

    print(f"  Saving to {file_path}...")
    # Save to parquet with compression
    df.to_parquet(file_path, compression="snappy")

    # Report file stats
    file_size_mb = os.path.getsize(file_path) / (1024 * 1024)
    memory_usage_mb = df.memory_usage(deep=True).sum() / (1024 * 1024)
    print(f"  Saved successfully: {len(df)} rows, {memory_usage_mb:.2f} MB in memory, {file_size_mb:.2f} MB on disk")

    return file_path

def load_position_data_from_parquet(
    experiment_name: str | None,
    period: str | None,
    data_dir: Path
) -> pd.DataFrame:
    """Loads saved position data from parquet files.

    Args:
        experiment_name (str, optional): Filter by experiment name. If None, load all experiments.
        period (str, optional): Filter by period (pre_social, social, post_social). If None, load all periods.
        data_dir (Path): Directory containing parquet files (default: "/nfs/nhome/live/apouget/ProjectAeon/aeon_methods_paper_tracking_data")

    Returns:
        pd.DataFrame: Combined DataFrame of all matching parquet files.
    """
    if not data_dir.exists():
        print(f"Directory {data_dir} does not exist. No position data files found.")
        return pd.DataFrame()

    # Create pattern based on filters
    pattern = ""
    if experiment_name:
        pattern += f"{experiment_name}_"
    else:
        pattern += "*_"

    if period:
        pattern += f"{period}_"
    else:
        pattern += "*_"

    pattern += "position.parquet"

    # Find matching files
    matching_files = list(data_dir.glob(pattern))

    if not matching_files:
        print(f"No matching position data files found with pattern: {pattern}")
        return pd.DataFrame()

    print(f"Found {len(matching_files)} matching files")

    # Load and concatenate matching files
    dfs = []
    total_rows = 0
    for file in matching_files:
        print(f"Loading {file}...")
        df = pd.read_parquet(file)
        total_rows += len(df)
        dfs.append(df)
        print(f"  Loaded {len(df)} rows")

    # Combine data
    if dfs:
        combined_df = pd.concat(dfs, ignore_index=True)
        combined_df = combined_df.set_index('time')
        print(f"Combined data: {len(combined_df)} rows")
        return combined_df
    else:
        return pd.DataFrame()

In [ ]:
# Directory to save / load parquet files
data_dir = Path("/ceph/aeon/aeon/code/scratchpad/methods_paper_data")
os.makedirs(data_dir, exist_ok=True)

In [15]:
"""Example usage: saving pos data"""

# Process all experiments and all time periods
for exp in experiments:
    key = {"experiment_name": exp["name"]}
    print(f"\nProcessing experiment: {exp['name']}")

    # Define time periods
    periods = {
        "pre_social": (exp["pre_social_start"], exp["pre_social_end"]),
        "social": (exp["social_start"], exp["social_end"]),
        "post_social": (exp["post_social_start"], exp["post_social_end"])
    }

    # Process each period
    for period_name, (period_start, period_end) in periods.items():
        print(f"\n  Loading {period_name} position data...")

        # Load position data for this period
        # position_df = load_position_data(key, period_start, period_end)
        position_df = load_position_data(
            key, period_start, str(pd.Timestamp(period_start) + pd.Timedelta("2h"))
        )

        if not position_df.empty:
            # Save to parquet
            save_position_data_to_parquet(
                position_df,
                exp["name"],
                period_name,
                data_dir
            )
        else:
            print(f"  No position data to save for {exp['name']} during {period_name} period")
        break


Processing experiment: social0.2-aeon3

  Loading pre_social position data...
  Querying data from 2024-01-31 11:00:00 to 2024-02-08 15:00:00...


  Retrieved 26653907 rows of position data
  Saving to /ceph/aeon/aeon/code/scratchpad/methods_paper_data/social0.2-aeon3_pre_social_position.parquet...
  Saved successfully: 26653907 rows, 8057.87 MB in memory, 496.87 MB on disk

Processing experiment: social0.2-aeon4

  Loading pre_social position data...
  Querying data from 2024-01-31 10:00:00 to 2024-02-08 15:00:00...
  Error loading position data for social0.2-aeon4 (2024-01-31 10:00:00 to 2024-02-08 15:00:00): No Chunk found between 2024-01-31 10:00:00 and 2024-02-08 15:00:00
  No position data to save for social0.2-aeon4 during pre_social period

Processing experiment: social0.3-aeon3

  Loading pre_social position data...
  Querying data from 2024-06-08 18:00:00 to 2024-06-17 13:00:00...
  Error loading position data for social0.3-aeon3 (2024-06-08 18:00:00 to 2024-06-17 13:00:00): No Chunk found between 2024-06-08 18:00:00 and 2024-06-17 13:00:00
  No position data to save for social0.3-aeon3 during pre_social period

Process

In [23]:
"""Example usage: loading pos data"""

# Example 1: Load all pre-social data
pre_social_df = load_position_data_from_parquet(
    experiment_name=None, period="pre_social", data_dir=data_dir
)
display(pre_social_df.head())

# Example 2: Load just one experiment's social data
social_exp3_df = load_position_data_from_parquet(
    experiment_name="social0.2-aeon3",
    period="pre_social",
    data_dir=data_dir
)
display(social_exp3_df.head())

Found 1 matching files
Loading /ceph/aeon/aeon/code/scratchpad/methods_paper_data/social0.2-aeon3_pre_social_position.parquet...
  Loaded 26653907 rows
Combined data: 26653907 rows


,experiment_name,identity_name,identity_likelihood,anchor_part,x,y,likelihood,period
time,,,,,,,,
2024-01-31 11:28:45.543520,social0.2-aeon3,BAA-1104045,NaN,anchor_spine2,1227.549316,467.023895,0.977126,pre_social
2024-01-31 11:28:51.000000,social0.2-aeon3,BAA-1104045,NaN,anchor_spine2,1227.453125,461.909515,0.994896,pre_social
2024-01-31 11:28:51.020000,social0.2-aeon3,BAA-1104045,NaN,anchor_spine2,1227.098145,458.853943,0.996563,pre_social
2024-01-31 11:28:51.040000,social0.2-aeon3,BAA-1104045,NaN,anchor_spine2,1226.879150,454.088440,0.983516,pre_social
2024-01-31 11:28:51.060000,social0.2-aeon3,BAA-1104045,NaN,anchor_spine2,1226.861572,451.292419,0.985476,pre_social


Found 1 matching files
Loading /ceph/aeon/aeon/code/scratchpad/methods_paper_data/social0.2-aeon3_pre_social_position.parquet...
  Loaded 26653907 rows
Combined data: 26653907 rows


,experiment_name,identity_name,identity_likelihood,anchor_part,x,y,likelihood,period
time,,,,,,,,
2024-01-31 11:28:45.543520,social0.2-aeon3,BAA-1104045,NaN,anchor_spine2,1227.549316,467.023895,0.977126,pre_social
2024-01-31 11:28:51.000000,social0.2-aeon3,BAA-1104045,NaN,anchor_spine2,1227.453125,461.909515,0.994896,pre_social
2024-01-31 11:28:51.020000,social0.2-aeon3,BAA-1104045,NaN,anchor_spine2,1227.098145,458.853943,0.996563,pre_social
2024-01-31 11:28:51.040000,social0.2-aeon3,BAA-1104045,NaN,anchor_spine2,1226.879150,454.088440,0.983516,pre_social
2024-01-31 11:28:51.060000,social0.2-aeon3,BAA-1104045,NaN,anchor_spine2,1226.861572,451.292419,0.985476,pre_social


---

---

### Sleep bouts

In [149]:
cm2px = 5.2

In [145]:
def excise_swaps(pos_df: pd.DataFrame, max_speed: float) -> pd.DataFrame:
    """Excises swaps in the position data.

    Args:
        pos_df (pd.DataFrame): DataFrame containing position data.
        max_speed (float): Maximum speed (px/s) threshold over which we assume a swap.

    Returns:
        pd.DataFrame: DataFrame with swaps excised.
    """
    dt = pos_df.index.diff().total_seconds()
    dx = pos_df["x"].diff()
    dy = pos_df["y"].diff()
    pos_df["inst_speed"] = np.sqrt(dx**2 + dy**2) / dt

    # Identify jumps
    jumps = pos_df["inst_speed"] > max_speed
    shift_down = jumps.shift(1)
    shift_down.iloc[0] = False
    shift_up = jumps.shift(-1)
    shift_up.iloc[len(jumps) - 1] = False
    jump_starts = jumps & ~shift_down
    jump_ends = jumps & ~shift_up
    jump_start_indices = np.where(jump_starts)[0]
    jump_end_indices = np.where(jump_ends)[0]

    if np.any(jumps):

        # Ensure the lengths match
        if len(jump_start_indices) > len(jump_end_indices):  # jump-in-progress at start
            jump_end_indices = np.append(jump_end_indices, len(pos_df) - 1)
        elif len(jump_start_indices) < len(jump_end_indices):  # jump-in-progress at end
            jump_start_indices = np.insert(jump_start_indices, 0, 0)

        # Excise jumps by setting speed to nan in jump regions
        for start, end in zip(jump_start_indices, jump_end_indices):  # removed strict=True
            pos_df.loc[pos_df.index[start]:pos_df.index[end], "inst_speed"] = np.nan
        pos_df.dropna(subset=["inst_speed"], inplace=True)

    return pos_df

In [ ]:
# Given pos_df and animal name, reutrn all sleep bouts in df within the pos_df time period

def sleep_bouts(
    pos_df: pd.DataFrame,
    animal_name: str,
    move_thresh: float = 5 * 5.2,  # 5 cm, in cm
    max_speed: float = 100 * 5.2,  # 100 cm/s, in px/s
) -> pd.DataFrame:
    """Returns sleep bouts for a given animal within the specified position data time period.

    Args:
        pos_df (pd.DataFrame): DataFrame containing position data.
        animal_name (str): Name of the animal to filter by.
        move_thresh (float): Movement (in px) threshold to define sleep bouts.
        max_speed (float): Maximum speed threshold for excising swaps (default: 100 cm/s in px/s).

    Returns:
        pd.DataFrame: DataFrame containing sleep bouts for the specified animal.
    """
    animal_data = pos_df[pos_df["identity_name"] == animal_name].copy()
    if animal_data.empty:
        print(f"No position data found for {animal_name}")
        return pd.DataFrame()

    # Set some constants and placeholder `windows_df` which will be combined into `bouts_df`
    sleep_win = pd.Timedelta("1m")
    sleep_windows_df = pd.DataFrame(columns=["animal_name", "start", "end", "duration", "period"])

    # Create time windows based on start and end time
    data_start_time = animal_data.index.min()
    data_end_time = animal_data.index.max()
    window_starts = pd.date_range(start=data_start_time, end=data_end_time, freq=sleep_win)

    # <s> Process each time window
    period = animal_data["period"].iloc[0]
    pbar = tqdm(window_starts, desc=f"Processing sleep bouts for {animal_name} in {period}")
    for win_start in pbar:
        win_end = win_start + sleep_win
        win_data = animal_data[
            (animal_data.index >= win_start) & (animal_data.index < win_end)
        ].copy()
        if len(win_data) < 100:  # skip windows with too little data
            continue

        # Excise id swaps (based on pos / speed jumps)
        win_data = excise_swaps(win_data, max_speed)

        # Calculate the displacement - maximum distance between any two points in the window
        dx = win_data["x"].max() - win_data["x"].min()
        dy = win_data["y"].max() - win_data["y"].min()
        displacement = np.sqrt(dx**2 + dy**2)

        # If displacement is less than threshold, consider it a sleep bout
        if displacement < move_thresh:
            new_bout = {
                "animal_name": animal_name,
                "start": win_start,
                "end": win_end,
                "duration": sleep_win,
                "period": win_data["period"].iloc[0]
            }
            sleep_windows_df = pd.concat([sleep_windows_df, pd.DataFrame([new_bout])], ignore_index=True)
    # </s>

    # <s> Now merge consecutive sleep windows into continuous bouts
    if sleep_windows_df.empty:
        return pd.DataFrame(columns=["animal_name", "start", "end", "duration", "period"])
    # Initialize the merged bouts dataframe with the first window
    sleep_bouts_df = pd.DataFrame([{
        "animal_name": animal_name,
        "start": sleep_windows_df.iloc[0]["start"],
        "end": sleep_windows_df.iloc[0]["end"],
        "duration": sleep_windows_df.iloc[0]["duration"],
        "period": sleep_windows_df.iloc[0]["period"]
    }])
    # Iterate through remaining windows and merge consecutive ones
    for i in range(1, len(sleep_windows_df)):
        current_window = sleep_windows_df.iloc[i]
        last_bout = sleep_bouts_df.iloc[-1]

        if current_window["start"] == last_bout["end"]:  # continue bout
            sleep_bouts_df.at[len(sleep_bouts_df)-1, "end"] = current_window["end"]
            sleep_bouts_df.at[len(sleep_bouts_df)-1, "duration"] = \
                sleep_bouts_df.iloc[-1]["end"] - sleep_bouts_df.iloc[-1]["start"]
        else:  # start a new bout
            new_bout = {
                "animal_name": animal_name,
                "start": current_window["start"],
                "end": current_window["end"],
                "duration": current_window["duration"],
                "period": current_window["period"]
            }
            sleep_bouts_df = pd.concat([sleep_bouts_df, pd.DataFrame([new_bout])], ignore_index=True)
    # </s>

    # Set min bout time
    min_bout_time = pd.Timedelta("2m")
    sleep_bouts_df = sleep_bouts_df[sleep_bouts_df["duration"] >= min_bout_time]

    return sleep_bouts_df

In [147]:
"""Example usage:"""

pos_df = pre_social_df
animal_name = pos_df["identity_name"].iloc[0]
sleep_bouts_df = sleep_bouts(pos_df.iloc[0:4_000_000], animal_name)


Processing sleep bouts for BAA-1104045 in pre_social:   0%|          | 0/1307 [00:00<?, ?it/s]

/tmp/ipykernel_1549225/1223152315.py:58: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  sleep_windows_df = pd.concat([sleep_windows_df, pd.DataFrame([new_bout])], ignore_index=True)


In [148]:
sleep_bouts_df

,animal_name,start,end,duration,period
0,BAA-1104045,2024-01-31 13:17:45.543520,2024-01-31 13:21:45.543520,0 days 00:04:00,pre_social
1,BAA-1104045,2024-01-31 14:33:45.543520,2024-01-31 14:35:45.543520,0 days 00:02:00,pre_social
7,BAA-1104045,2024-01-31 15:44:45.543520,2024-01-31 15:46:45.543520,0 days 00:02:00,pre_social
9,BAA-1104045,2024-01-31 16:15:45.543520,2024-01-31 16:17:45.543520,0 days 00:02:00,pre_social
10,BAA-1104045,2024-01-31 16:28:45.543520,2024-01-31 16:30:45.543520,0 days 00:02:00,pre_social
12,BAA-1104045,2024-01-31 17:46:45.543520,2024-01-31 17:48:45.543520,0 days 00:02:00,pre_social
13,BAA-1104045,2024-01-31 18:13:45.543520,2024-01-31 18:18:45.543520,0 days 00:05:00,pre_social
16,BAA-1104045,2024-01-31 19:19:45.543520,2024-01-31 19:25:45.543520,0 days 00:06:00,pre_social
17,BAA-1104045,2024-01-31 19:36:45.543520,2024-01-31 19:42:45.543520,0 days 00:06:00,pre_social
18,BAA-1104045,2024-01-31 19:44:45.543520,2024-01-31 21:19:45.543520,0 days 01:35:00,pre_social


### Explore bouts

In [150]:
# Given pos_df, animal name, nest xy, reutrn all explore bouts in df

nest_center = np.array((1215, 530))
cm2px = 5.2
nest_radius = 14 * cm2px  # 14 cm, in px

def explore_bouts(
    pos_df: pd.DataFrame,
    animal_name: str,
    nest_center: np.ndarray,
    nest_radius: float = 14 * 5.2,  # 14 cm, in px
    max_speed: float = 100 * 5.2,  # 100 cm/s, in px/s
) -> pd.DataFrame:
    """Returns exploration bouts for a given animal within the specified position data time period.

    Args:
        pos_df (pd.DataFrame): DataFrame containing position data.
        animal_name (str): Name of the animal to filter by.
        nest_center (np.ndarray): Coordinates of the nest center.
        nest_radius (float): Radius of the nest area (default: 14 cm in px).
        max_speed (float): Maximum speed threshold for excising swaps (default: 100 cm/s in px/s).

    Returns:
        pd.DataFrame: DataFrame containing exploration bouts for the specified animal.
    """
    animal_data = pos_df[pos_df["identity_name"] == animal_name].copy()
    if animal_data.empty:
        print(f"No position data found for {animal_name}")
        return pd.DataFrame()

    # Set some constants and placeholder `windows_df` which will be combined into `bouts_df`
    explore_win = pd.Timedelta("1m")
    explore_windows_df = pd.DataFrame(columns=["animal_name", "start", "end", "duration", "period"])

    # Create time windows based on start and end time
    data_start_time = animal_data.index.min()
    data_end_time = animal_data.index.max()
    window_starts = pd.date_range(start=data_start_time, end=data_end_time, freq=explore_win)

    # <s> Process each time window (use tqdm for progress bar)
    period = animal_data["period"].iloc[0]
    pbar = tqdm(window_starts, desc=f"Processing explore bouts for {animal_name} in {period}")
    for win_start in pbar:
        win_end = win_start + explore_win
        win_data = animal_data[
            (animal_data.index >= win_start) & (animal_data.index < win_end)
        ].copy()
        if len(win_data) < 100:  # skip windows with too little data
            continue

        # Excise id swaps (based on pos / speed jumps)
        win_data = excise_swaps(win_data, max_speed)

        # If majority of time in a window is outside nest, consider it an explore bout
        dx = win_data["x"] - nest_center[0]
        dy = win_data["y"] - nest_center[1]
        distance_from_nest = np.sqrt(dx**2 + dy**2)
        frac_out_nest = (distance_from_nest > nest_radius).sum() / len(win_data)
        if frac_out_nest > 0.5:
            new_bout = {
                "animal_name": animal_name,
                "start": win_start,
                "end": win_end,
                "duration": explore_win,
                "period": win_data["period"].iloc[0]
            }
            explore_windows_df = pd.concat([explore_windows_df, pd.DataFrame([new_bout])], ignore_index=True)
    # </s>

    # <s> Now merge consecutive explore windows into continuous bouts
    if explore_windows_df.empty:
        return pd.DataFrame(columns=["animal_name", "start", "end", "duration", "period"])
    # Initialize the merged bouts dataframe with the first window
    explore_bouts_df = pd.DataFrame([{
        "animal_name": animal_name,
        "start": explore_windows_df.iloc[0]["start"],
        "end": explore_windows_df.iloc[0]["end"],
        "duration": explore_windows_df.iloc[0]["duration"],
        "period": explore_windows_df.iloc[0]["period"]
    }])
    # Iterate through remaining windows and merge consecutive ones
    for i in range(1, len(explore_windows_df)):
        current_window = explore_windows_df.iloc[i]
        last_bout = explore_bouts_df.iloc[-1]

        if current_window["start"] == last_bout["end"]:  # continue bout
            explore_bouts_df.at[len(explore_bouts_df)-1, "end"] = current_window["end"]
            explore_bouts_df.at[len(explore_bouts_df)-1, "duration"] = \
                explore_bouts_df.iloc[-1]["end"] - explore_bouts_df.iloc[-1]["start"]
        else:  # start a new bout
            new_bout = {
                "animal_name": animal_name,
                "start": current_window["start"],
                "end": current_window["end"],
                "duration": current_window["duration"],
                "period": current_window["period"]
            }
            explore_bouts_df = pd.concat([explore_bouts_df, pd.DataFrame([new_bout])], ignore_index=True)
    # </s>

    return explore_bouts_df

In [151]:
"""Example usage:"""

pos_df = pre_social_df
animal_name = pos_df["identity_name"].iloc[0]
explore_bouts_df = explore_bouts(pos_df.iloc[0:4_000_000], animal_name, nest_center)
explore_bouts_df

Processing explore bouts for BAA-1104045 in pre_social:   0%|          | 0/1307 [00:00<?, ?it/s]

/tmp/ipykernel_1549225/707162469.py:67: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  explore_windows_df = pd.concat([explore_windows_df, pd.DataFrame([new_bout])], ignore_index=True)


,animal_name,start,end,duration,period
0,BAA-1104045,2024-01-31 11:28:45.543520,2024-01-31 12:04:45.543520,0 days 00:36:00,pre_social
1,BAA-1104045,2024-01-31 12:05:45.543520,2024-01-31 12:16:45.543520,0 days 00:11:00,pre_social
2,BAA-1104045,2024-01-31 12:17:45.543520,2024-01-31 12:20:45.543520,0 days 00:03:00,pre_social
3,BAA-1104045,2024-01-31 12:21:45.543520,2024-01-31 12:28:45.543520,0 days 00:07:00,pre_social
4,BAA-1104045,2024-01-31 12:29:45.543520,2024-01-31 12:37:45.543520,0 days 00:08:00,pre_social
5,BAA-1104045,2024-01-31 12:38:45.543520,2024-01-31 12:47:45.543520,0 days 00:09:00,pre_social
6,BAA-1104045,2024-01-31 12:48:45.543520,2024-01-31 12:58:45.543520,0 days 00:10:00,pre_social
7,BAA-1104045,2024-01-31 13:00:45.543520,2024-01-31 13:03:45.543520,0 days 00:03:00,pre_social
8,BAA-1104045,2024-01-31 13:05:45.543520,2024-01-31 13:08:45.543520,0 days 00:03:00,pre_social
9,BAA-1104045,2024-01-31 13:09:45.543520,2024-01-31 13:24:45.543520,0 days 00:15:00,pre_social
